# 01 - Data Exploration & Preparation
**Customer Intelligence System - المرحلة الأولى: Data Preparation**

الخطوات:
1. قراءة الملفات الستة
2. فحص الجودة (الحجم، الأنواع، الناقص، المكرر)
3. تحويل أنواع التواريخ
4. فحص سلامة مفاتيح الربط (Referential Integrity)
5. بناء الجدول الرئيسي `master_orders` وحفظه في `outputs/`

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_DIR = Path("../data")
OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

names = ["date_dim", "deliveryman_dim", "fact_orders", "product_dim", "store_dim", "user_dim"]
dfs = {n: pd.read_csv(DATA_DIR / f"{n}.csv") for n in names}

for n, df in dfs.items():
    print(f"{n:16s} {df.shape[0]:>7,} rows x {df.shape[1]} cols")

## 1) فحص الجودة لكل جدول

In [ ]:
for n, df in dfs.items():
    print(f"\n===== {n} =====")
    print(df.dtypes.to_string())
    nulls = df.isna().sum()
    print("Nulls     :", nulls[nulls > 0].to_dict() or "none")
    print("Duplicates:", df.duplicated().sum())
    display(df.head(3))

## 2) تحويل أنواع التواريخ
التواريخ مقروءة كـ نص (string) ولازم تتحول لـ datetime.

In [ ]:
fact = dfs["fact_orders"].copy()
users = dfs["user_dim"].copy()
dates = dfs["date_dim"].copy()

fact["order_date"] = pd.to_datetime(fact["order_date"])
fact["delivery_date"] = pd.to_datetime(fact["delivery_date"])
users["signup_date"] = pd.to_datetime(users["signup_date"])
dates["date"] = pd.to_datetime(dates["date"])

print(fact[["order_date", "delivery_date"]].dtypes)

## 3) مستوى التفصيل (Grain) لجدول `fact_orders`
`order_id` بيتكرر، يعني الجدول على مستوى **بند داخل الطلب (order line)** مش طلب كامل.
كل صف = منتج واحد جوه طلب. لازم ناخد ده في الاعتبار عند حساب عدد الطلبات.

In [ ]:
print("Rows (order lines)  :", len(fact))
print("Distinct order_id   :", fact["order_id"].nunique())
print("Items per order     :")
print(fact.groupby("order_id").size().describe().round(2).to_string())

# هل الخصم والحالة والتاريخ ثابتين جوه الطلب الواحد؟
per_order = fact.groupby("order_id")[["user_id", "store_id", "order_date", "order_status", "discount"]].nunique()
print("\nأعمدة بتتغير جوه نفس الطلب (المفروض صفر):")
print((per_order > 1).sum().to_string())

## 4) سلامة مفاتيح الربط (Orphan Keys)

In [ ]:
checks = [
    ("user_id",  dfs["user_dim"]["user_id"]),
    ("item_id",  dfs["product_dim"]["item_id"]),
    ("store_id", dfs["store_dim"]["store_id"]),
    ("date_key", dfs["date_dim"]["date_key"]),
]
for key, ref in checks:
    print(f"{key:10s} orphans in fact_orders:", (~fact[key].isin(ref)).sum())

dm = fact["deliveryman_id"].dropna()
print(f"{'deliveryman_id':10s} orphans (non-null)   :", (~dm.isin(dfs['deliveryman_dim']['deliveryman_id'])).sum())

# المفاتيح لازم تكون unique في جداول الأبعاد
for n, k in [("user_dim", "user_id"), ("product_dim", "item_id"), ("store_dim", "store_id"),
             ("deliveryman_dim", "deliveryman_id"), ("date_dim", "date_key")]:
    assert dfs[n][k].is_unique, f"{n}.{k} not unique"
print("\nكل مفاتيح الأبعاد unique ✅")

## 5) القيم الناقصة المتوقعة (مش أخطاء)
- `delivery_date` فاضي لما الحالة `Cancelled` أو `Returned`.
- `deliveryman_id` فاضي لأن معظم الطلبات من الفروع (Offline). المتاجر الأونلاين (`S-07`, `S-08`) هي اللي ليها مندوب توصيل.

In [ ]:
print("delivery_date null by status:")
print(fact[fact["delivery_date"].isna()]["order_status"].value_counts().to_string())

print("\nنسبة الصفوف اللي ليها مندوب، حسب المتجر:")
print(fact.groupby("store_id")["deliveryman_id"].apply(lambda s: round(s.notna().mean(), 3)).to_string())

print("\nمنطقية التواريخ:")
print("delivery قبل order :", (fact["delivery_date"] < fact["order_date"]).sum())
print("order قبل signup   :", (fact.merge(users[["user_id", "signup_date"]], on="user_id")
                                  .eval("order_date < signup_date")).sum())

## 6) بناء الجدول الرئيسي `master_orders`
بنربط `fact_orders` بالأبعاد الأربعة بـ `left join` وبنتأكد إن عدد الصفوف ما اتغيرش.

In [ ]:
master = (
    fact
    .merge(users,                       on="user_id",  how="left")
    .merge(dfs["product_dim"],          on="item_id",  how="left")
    .merge(dfs["store_dim"],            on="store_id", how="left")
    .merge(dfs["deliveryman_dim"].rename(columns={"zone": "delivery_zone", "type": "deliveryman_type"}),
           on="deliveryman_id", how="left")
    .merge(dates.drop(columns=["date"]), on="date_key", how="left")
)

assert len(master) == len(fact), "الربط غيّر عدد الصفوف!"

# أعمدة مشتقة هتفيد في التحليل
master["is_delivered"]      = master["order_status"].eq("Delivered")
master["has_discount"]      = master["discount"] > 0
master["delivery_days"]     = (master["delivery_date"] - master["order_date"]).dt.days
master["order_dow"]         = master["order_date"].dt.day_name()   # الداتا فيها تاريخ بس، مفيش ساعة الطلب

print(master.shape)
display(master.head())

In [ ]:
master.to_csv(OUT_DIR / "master_orders.csv", index=False)
print("Saved ->", OUT_DIR / "master_orders.csv")

## ملاحظات مهمة قبل التحليل
1. **Grain:** `fact_orders` = order line. للـ order-level لازم `groupby("order_id")`.
2. **الخصم (`discount`)** ثابت جوه الطلب الواحد، قيمه: 0 / 0.05 / 0.10 / 0.15، وحوالي 50% من الطلبات عليها خصم بنسبة ثابتة تقريبًا كل شهر. يعني **مفيش عمود بيقول فترة العرض**؛ لو عايزين نحدد "وقت العروض" لازم نعرّفه (مثلًا أيام الخصم فوق المتوسط).
3. **`sales_amount`:** محتاجين نتأكد من الفريق هل هو قبل الخصم ولا بعده.
4. **`Cancelled` / `Returned`** لازم تتفلتر أو تتعامل معاها بشكل منفصل في تحليل المشتريات الفعلية.
5. **`date_dim`** بيغطي من 2025-10-01 لـ 2026-09-30 (سنة واحدة).